In [2]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

In [3]:
sentiment = pd.read_csv("../data/processed/24to26_pospayreviews.csv")
sentiment.head()

,review_id,text,text_clean,rating,review_date,sentiment,issues
0,2a789331-f2b1-45a9-8f7c-0a3f22168238,ENTAH DARI KURIR ENTAH DARI KARYAWAN ATAUPUN D...,entah dari kurir entah dari karyawan ataupun d...,1,2026-04-16,negative,"['pelayanan kantor pos buruk', 'kurir/karyawan..."
1,1c80dbda-189a-46f3-bd84-19ddcaa356b5,bgus,bgus,5,2026-04-15,positive,[]
2,7bfb8ff1-326e-46c0-b6e4-113ab4528b7b,"Sangat nyaman dan memuaskan, transaksi nya lan...","sangat nyaman dan memuaskan, transaksi nya lan...",5,2026-04-14,positive,[]
3,75121e2c-dde8-4ec6-9a40-2cb735e5bec7,aplikasinya rekomen banget buat pemula yang ma...,aplikasinya rekomen banget buat pemula yang ma...,5,2026-04-13,positive,[]
4,8de6e3a5-39c2-4b1b-8686-ff0831d9a173,mau login selalu muncul konfigurasi.. tah pape...,mau login selalu muncul konfigurasi.. tah pape...,1,2026-04-13,negative,['masalah login muncul konfigurasi yang tidak ...


In [4]:
df = sentiment.copy()

In [5]:
df["Month"] = pd.to_datetime(df["review_date"]).dt.to_period("M")

trend = df.groupby("Month")["sentiment"].value_counts(normalize=False).unstack()

trend.index = trend.index.astype(str)
trend["total"] = trend[["positive", "neutral", "negative"]].sum(axis=1)

trend_pct = trend[["positive", "neutral", "negative"]].div(trend["total"], axis=0) * 100
trend["month"] = pd.to_datetime(trend.index).strftime("%b")

In [6]:
import plotly.express as px

rating_counts = df["rating"].value_counts().sort_index()

fig = px.bar(
    x=rating_counts.index,
    y=rating_counts.values,
    title="<b>2026 Pospay's Google Play Rating Distribution</b>",
    text=rating_counts.values
)

fig.update_traces(
    width=0.8  # 👈 CONTROL WIDTH HERE
)

fig.update_layout(
    xaxis_title="Rating (Stars)",
    yaxis_title="Number of Reviews",
    template="plotly_white"
)

fig.show()

In [7]:
fig = px.histogram(df, x="sentiment", title="<b>2026 Pospay's Google Play Sentiment Distribution</b>", text_auto=True)
#edit xaxis and yaxis title
fig.update_layout(
    xaxis_title="Sentiment",
    yaxis_title="Number of Reviews",
    template="plotly_white"
)
fig.show()

In [10]:
import pandas as pd
import plotly.graph_objects as go


# ===== PREP DATA =====
trend.index = pd.to_datetime(trend.index)
trend["month"] = trend.index.strftime("%b")
trend["month_year"] = trend.index.strftime("%b %Y")

trend["total"] = trend[["positive", "neutral", "negative"]].sum(axis=1)

trend_pct = trend[["positive", "neutral", "negative"]].div(trend["total"], axis=0) * 100


# ===== BUILD FIGURE =====
fig = go.Figure()

color = px.colors.qualitative.Plotly
colors = {
    "negative": "red",
    "positive": "green",
    "neutral": "blue"
}
# 🔹 GROUPED BARS (VISIBLE)
for s in ["positive", "negative", "neutral"]:
    fig.add_bar(
        x=trend["month_year"],
        y=trend[s],
        name=s.capitalize(),
        marker=dict(color=colors[s], opacity=0.4),

        text=[f"{v:.1f}%" for v in trend_pct[s]],

        textposition="inside",

        textfont=dict(size=13, color="white")
    )

line_colors = {"negative": "red", "positive": "green", "neutral": "darkblue"}


for s in ["negative", "positive", "neutral"]:
    fig.add_scatter(
        x=trend["month_year"],
        y=trend[s],
        mode="lines+markers",
        name=f"{s.capitalize()} Trend",
        line=dict(color=line_colors[s], width=3)
    )


# 🔥 TOTAL (FIXED POSITION)
fig.add_scatter(
    x=trend["month_year"],
    y=trend["positive"] * 1.1,
    mode="text",
    text=[f"<b>Total Review: {v}</b>" for v in trend["total"]],
    textposition="top center",
    showlegend=False
)

# ===== LAYOUT =====
fig.update_layout(
    barmode="overlay",  # 👈 FIXED
    title="<b>Trend Review per Bulan</b>",
    xaxis_title="Month",
    yaxis_title="Count",
    template="plotly_white"
)

fig.show()

In [11]:
from collections import Counter

negative_reviews = df[df["sentiment"] == "negative"]
negative_issue_counts = Counter(negative_reviews["issues"].explode())

top_issues = negative_issue_counts.most_common(10)
top_issues

[("['aplikasi tidak bisa dibuka']", 28),
 ("['tidak bisa login']", 24),
 ("['aplikasi tidak jelas']", 18),
 ("['aplikasi sering error']", 16),
 ("['tidak bisa cek BSU']", 16),
 ("['aplikasi buruk']", 14),
 ("['aplikasi tidak berguna']", 12),
 ("['kualitas aplikasi buruk']", 10),
 ("['aplikasi lemot']", 9),
 ("['aplikasi jelek']", 9)]

In [12]:
negative_reviews["issues"].explode().value_counts().head(20)

issues
['aplikasi tidak bisa dibuka']          28
['tidak bisa login']                    24
['aplikasi tidak jelas']                18
['aplikasi sering error']               16
['tidak bisa cek BSU']                  16
['aplikasi buruk']                      14
['aplikasi tidak berguna']              12
['kualitas aplikasi buruk']             10
['aplikasi lemot']                       9
['aplikasi jelek']                       9
['tidak bisa daftar']                    9
['sulit login']                          8
['aplikasi ribet']                       8
['aplikasi lambat']                      8
['sering error']                         7
['jenis bantuan tidak bisa diklik']      7
['aplikasi rusak']                       6
['tidak bisa membuat akun']              6
['akun diblokir tanpa alasan jelas']     6
[]                                       6
Name: count, dtype: int64

In [18]:
CATEGORIES = [
    "authentication",
    "transaction",
    "security",       # NEW
    "feature",        # NEW
    "performance",
    "ux/ui",
    "support",
    "pricing",        # NEW
    "other"
]

def map_issue(issue):
    issue = issue.lower()

    # 🔴 Authentication
    if any(k in issue for k in [
        "login", "password", "access", "verification", "email", "face", "akun", "validasi", "upgrade", "PIN", "username", "credentials", "otp", "two-factor", "2fa", "biometric", "sandi"
    ]):
        return "authentication"

    # 🔴 Security (NEW)
    if any(k in issue for k in [
        "fraud", "unauthorized", "tidak sah", "keamanan", "uang berkurang", "account ballance", "phishing", "scam", "hacker", "breach", "data leak", "privacy", "encryption", "vulnerability", "privasi", "enkripsi", "kerentanan"
    ]):
        return "security"

    # 🔴 Transaction
    if any(k in issue for k in [
        "transfer", "balance", "pulsa", "topup", "payment", "transaksi", "refund", "withdrawal", "deposit", "billing", "invoice", "bank transfer", "balance", "pulsa", "topup", "transaksi", "refund", "withdrawal", "deposit", "billing", "invoice", "bank transfer", "paket", "meterai", "ewallet"
    ]):
        return "transaction"

    # 🔴 Feature availability (NEW)
    if any(k in issue for k in [
        "cannot", "tidak bisa", "tidak muncul", "not available", "fitur", "menu", "tokopedia", "section", "option", "fitur hilang", "fitur tidak muncul", "fitur tidak tersedia"
    ]):
        return "feature"

    # 🟠 Performance
    if any(k in issue for k in [
        "slow", "lemot", "loading", "unresponsive", "always problematic", "frequent", "gangguan", "disruption", "server", "error", "crash", "hang", "lag", "timeout", "bug", "glitch", "freeze", "restart", "update", "upgrade"
    ]):
        return "performance"

    # 🟠 UX/UI
    if any(k in issue for k in [
        "difficult", "complicated", "ribet", "not user-friendly", "ui", "appearance", "branding", "design", "navigasi", "interface", "pusing", "bingung", "tidak intuitif", "tampilan", "desain", "navigasi", "antarmuka"
    ]):
        return "ux/ui"


    # 🟡 Support
    if any(k in issue for k in [
        "support", "service", "customer", "help", "cs", "contact", "response", "respon", "tanggapan", "bantuan", "layanan", "pelayanan", "dukungan", "komplain", "keluhan"
    ]):
        return "support"

    # 🟡 Pricing
    if any(k in issue for k in [
        "biaya", "fee", "expensive", "topup", "charge", "harga", "price", "topup", "biaya", "fee", "expensive", "charge", "harga", "price"
    ]):
        return "pricing"

    return "other"

df_exploded = negative_reviews.explode("issues")
df_exploded["category"] = df_exploded["issues"].apply(map_issue)

df_exploded["category"].value_counts()

category
authentication    1054
other              686
feature            641
performance        275
transaction        224
support             67
ux/ui               52
security            17
pricing             14
Name: count, dtype: int64

In [19]:
import pandas as pd

# =========================
# 1. EXPLODE (if not already)
# =========================
# df_exploded = negative_reviews.explode("issues")
# df_exploded["category"] = df_exploded["issues"].apply(map_issue)


# =========================
# 2. COUNT PER CATEGORY
# =========================
category_counts = (
    df_exploded["category"]
    .value_counts()
    .reset_index()
)
category_counts.columns = ["category", "count"]


# =========================
# 3. GET TOP ISSUES PER CATEGORY
# =========================
example_df = (
    df_exploded.groupby(["category", "issues"])
    .size()
    .reset_index(name="freq")
    .sort_values(["category", "freq"], ascending=[True, False])
)


# =========================
# 4. TAKE TOP 3 EXAMPLES
# =========================
top_examples = (
    example_df.groupby("category")
    .head(2)
    .groupby("category")["issues"]
    .apply(lambda x: "; ".join(x))
    .reset_index()
    .rename(columns={"issues": "examples"})
)


# =========================
# 5. MERGE COUNT + EXAMPLES
# =========================
summary_df = pd.merge(category_counts, top_examples, on="category")

summary_df = summary_df[["category", "examples", "count"]]

# Optional: sort by importance
summary_df = summary_df.sort_values("count", ascending=False)


# =========================
# 6. ADD PERCENTAGE (OPTIONAL)
# =========================
summary_df["percentage"] = (
    summary_df["count"] / summary_df["count"].sum() * 100
).round(1)



summary_df.to_csv("../data/processed/issue_summary.csv", index=False)

In [20]:
df_exploded[df_exploded["category"] == "other"]

,review_id,text,text_clean,rating,review_date,sentiment,issues,Month,category
5,bd6dd8a5-b6ad-4b6d-be71-154b5ef45dcd,"keranjang gak berfungsi, tiap d masukkan keran...","keranjang gak berfungsi, tiap d masukkan keran...",2,2026-04-13,negative,"['keranjang tidak berfungsi', 'item yang dimas...",2026-04,other
19,9e2dadc3-308e-4937-a245-0edd2f0899a4,kok beda sama yg di gambar,kok beda sama yg di gambar,2,2026-04-02,negative,['aplikasi berbeda dengan gambar'],2026-04,other
22,36c5e33a-1c84-40c6-a66b-238cde103664,makin ruwet sulit,makin ruwet sulit,1,2026-03-31,negative,['aplikasi makin rumit dan sulit digunakan'],2026-03,other
31,4f7b1355-4dce-49da-af72-d4702000fd03,SAYA DITIPU PAKE APLIKASI INI,saya ditipu pake aplikasi ini,1,2026-03-15,negative,['penipuan'],2026-03,other
32,900dd85e-5b07-4124-9b6f-3da5d7999407,tolong diperbaiki untuk qrisnya lama sekali ma...,tolong diperbaiki untuk qrisnya lama sekali ma...,1,2026-03-15,negative,['QRIS lama masuk'],2026-03,other
...,...,...,...,...,...,...,...,...,...
6794,6ea3e091-686b-4f5b-9ac0-19e0969b2ea0,Toup up ewallet kok gak ada pilihan,toup up ewallet kok gak ada pilihan,1,2024-01-15,negative,['tidak ada pilihan top up e-wallet'],2024-01,other
6809,85523e1a-cd13-4858-b439-e40fce702dad,Buruk 2x pembubuhan materai tidak sesuai yg di...,buruk 2x pembubuhan materai tidak sesuai yg di...,1,2024-01-08,negative,"['pembubuhan materai tidak sesuai', 'kerugian ...",2024-01,other
6812,15f88c80-f3c8-4f09-ac63-da1728f9581d,aplikasi bikinan negara kok bobrok sih,aplikasi bikinan negara kok bobrok sih,1,2024-01-05,negative,['kualitas aplikasi buruk'],2024-01,other
6815,f45322d9-3653-4662-9e57-22bab62c9a0d,"Mau ganti device ribet pake bangettttttt, kena...","mau ganti device ribet pake bangettttttt, kena...",1,2024-01-04,negative,['proses ganti device rumit dan tidak user fri...,2024-01,other
